# 05 · Feature Engineering — do the engineered features earn their keep?

`04_preprocessing` introduced four engineered columns: `amount_log`, `amount_robust`, `hour_sin`, `hour_cos`. This notebook **tests honestly whether they lift PR-AUC** on a held-out validation set, against the raw `V1..V28 + Amount + Time`.

We test on **two model families** because the answer can differ:
- **Linear (logistic regression).** Can't bend a curve; can't "see" a cyclical encoding as cyclic — to it, `(hour_sin, hour_cos)` is just two more axes.
- **Gradient-boosted trees (XGBoost).** Can split on smooth features and exploit interactions; usually the one to benefit from engineered numerics.

Whichever wins, the **next** notebook (`06_ml_prediction`) inherits that choice via `build_preprocessor()`.

In [ ]:
# project-root bootstrap — portable across VS Code / Jupyter / CLI
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / 'config' / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('working dir:', Path.cwd())

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

from fraud.io import read_parquet
from fraud.features import CyclicalHour, AmountTransforms
from fraud.preprocess import V_COLS

warnings.filterwarnings('ignore', category=UserWarning)
plt.rcParams.update({'figure.facecolor':'white','axes.spines.top':False,
                     'axes.spines.right':False,'axes.titleweight':'bold'})
FIG = Path('reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

train = read_parquet('data/processed/train.parquet')
valid = read_parquet('data/processed/valid.parquet')
train.shape, valid.shape

## 1 · The engineered features

- `log_amount = log1p(Amount)` — compresses the long $-tail.
- `(hour_sin, hour_cos) = (sin, cos)(2π · hour/24)` — encodes 23:00 and 00:00 as neighbours on a circle, which a model otherwise cannot know.

In [ ]:
def engineer(df):
    ch = CyclicalHour().fit_transform(df[['Time']])
    at = AmountTransforms().fit_transform(df[['Amount']])
    out = df[V_COLS].copy()
    out['log_amount'] = at[:, 0]
    out['hour_sin']   = ch[:, 0]
    out['hour_cos']   = ch[:, 1]
    return out

engineer(train).head()

In [ ]:
hour = ((train['Time'] / 3600) % 24).values
ang  = 2 * np.pi * hour / 24
sample = np.random.default_rng(0).choice(len(hour), size=4000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sc = axes[0].scatter(np.sin(ang[sample]), np.cos(ang[sample]),
                     c=hour[sample], cmap='twilight', s=4)
axes[0].set(title='hour → (sin, cos) cyclical encoding',
            xlabel='hour_sin', ylabel='hour_cos')
axes[0].set_aspect('equal')
fig.colorbar(sc, ax=axes[0], label='hour-of-day')

axes[1].hist(train['Amount'].clip(upper=2000), bins=60, alpha=.6, color='#3498db', label='Amount (raw, clipped)')
axes[1].hist(np.log1p(train['Amount']),         bins=60, alpha=.6, color='#27ae60', label='log1p(Amount)')
axes[1].set(title='Amount — raw vs log1p', xlabel='value', ylabel='count')
axes[1].legend()
fig.suptitle('Feature engineering — visual sanity check', fontweight='bold')
fig.tight_layout()
fig.savefig(FIG/'feature_engineering_cyclical.png', dpi=120, bbox_inches='tight')
plt.show()

## 2 · Test 1 — linear model (logistic regression)

Logistic regression cannot model interactions or curvature. Cyclical encodings give it extra axes but no extra leverage — we expect the engineered set to roughly tie or slightly hurt.

In [ ]:
def pr_auc_logreg(Xtr, ytr, Xva, yva):
    m = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight='balanced'))
    m.fit(Xtr, ytr)
    return float(average_precision_score(yva, m.predict_proba(Xva)[:, 1]))

X_raw_tr,  X_raw_va  = train[V_COLS + ['Amount','Time']], valid[V_COLS + ['Amount','Time']]
X_eng_tr,  X_eng_va  = engineer(train),                   engineer(valid)
y_tr, y_va = train.Class, valid.Class

logreg_raw = pr_auc_logreg(X_raw_tr, y_tr, X_raw_va, y_va)
logreg_eng = pr_auc_logreg(X_eng_tr, y_tr, X_eng_va, y_va)
print(f'logreg  raw       = {logreg_raw:.4f}')
print(f'logreg  engineered= {logreg_eng:.4f}')
print(f'logreg  lift      = {logreg_eng - logreg_raw:+.4f}')

## 3 · Test 2 — gradient-boosted model (XGBoost)

Trees split on smooth features and combine them via depth — the right setting to exploit `log_amount` and the `(sin, cos)` pair if there's anything there. We use the same `scale_pos_weight` as `04_preprocessing` derived (~577).

In [ ]:
spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

def pr_auc_xgb(Xtr, ytr, Xva, yva):
    m = XGBClassifier(scale_pos_weight=spw, eval_metric='aucpr',
                      n_estimators=300, max_depth=5, learning_rate=0.1,
                      tree_method='hist', n_jobs=-1, random_state=0)
    m.fit(Xtr, ytr)
    return float(average_precision_score(yva, m.predict_proba(Xva)[:, 1]))

xgb_raw = pr_auc_xgb(X_raw_tr, y_tr, X_raw_va, y_va)
xgb_eng = pr_auc_xgb(X_eng_tr, y_tr, X_eng_va, y_va)
print(f'xgb     raw       = {xgb_raw:.4f}')
print(f'xgb     engineered= {xgb_eng:.4f}')
print(f'xgb     lift      = {xgb_eng - xgb_raw:+.4f}')

## 4 · Verdict — table and bar chart

We report **PR-AUC** (not ROC-AUC, not accuracy) on validation for both model families, and the absolute lift from adding engineered features.

In [ ]:
summary = pd.DataFrame([
    {'model':'logreg',  'raw': logreg_raw, 'engineered': logreg_eng, 'lift': logreg_eng - logreg_raw},
    {'model':'xgboost', 'raw': xgb_raw,    'engineered': xgb_eng,    'lift': xgb_eng - xgb_raw},
]).round(4)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(summary))
w = 0.35
axes[0].bar(x - w/2, summary['raw'],        w, label='raw',        color='#7f8c8d')
axes[0].bar(x + w/2, summary['engineered'], w, label='engineered', color='#27ae60')
axes[0].set_xticks(x); axes[0].set_xticklabels(summary['model'])
axes[0].set(title='PR-AUC on validation', ylabel='PR-AUC', ylim=(0, 1))
axes[0].legend()
for i, (r, e) in enumerate(zip(summary['raw'], summary['engineered'])):
    axes[0].text(i - w/2, r + 0.01, f'{r:.3f}', ha='center', fontsize=9)
    axes[0].text(i + w/2, e + 0.01, f'{e:.3f}', ha='center', fontsize=9)

colors = ['#c0392b' if v < 0 else '#27ae60' for v in summary['lift']]
axes[1].bar(summary['model'], summary['lift'], color=colors)
axes[1].axhline(0, color='#333', lw=.6)
axes[1].set(title='Lift from engineered features', ylabel='Δ PR-AUC')
for i, v in enumerate(summary['lift']):
    axes[1].text(i, v + (0.001 if v >= 0 else -0.003), f'{v:+.4f}', ha='center', fontsize=9)

fig.suptitle('Feature-engineering A/B — raw vs engineered, two model families', fontweight='bold')
fig.tight_layout()
fig.savefig(FIG/'feature_engineering_results.png', dpi=120, bbox_inches='tight')
plt.show()
summary

**Decision recorded:**

- On a **linear** model the engineered features typically tie or slightly hurt — PCA features already capture the signal a linear decision boundary can use, and the cyclical encoding adds dimensions without leverage.
- On the **tree** model the engineered features either add a small lift or are roughly neutral. Even when neutral on PR-AUC, they are kept because:
  1. `log_amount` and `hour_sin/cos` are **interpretable** — they show up in SHAP plots and let an analyst tell the story ("flagged because the amount is high *and* it's 3am").
  2. The cost is one preprocessor call; the risk of carrying them is zero.

`06_ml_prediction` therefore uses the full `build_preprocessor()` output, then tunes XGBoost and picks a business-cost-optimal threshold.